# សំណើផ្លូវដំណើរកំសាន្តជាមួយការការគ្រប់គ្រងសមម័យជាមួយគ្នា

សៀវភៅតំណរនេះបង្ហាញពី **ការគ្រប់គ្រងសមម័យជាមួយគ្នា** ដោយប្រើ Microsoft Agent Framework។ យើងនឹងបង្កើតប្រព័ន្ធសំណើផ្លូវដំណើរកំសាន្តជាមួយភ្នាក់ងារបីដែលមានជំនាញជាក់លាក់ដែលធ្វើការជាមួយគ្នា ប្រកបដោយសម័យដើម្បីផ្តល់នូវការយល់ដឹងដំណើរកំសាន្តយ៉ាងទូលំទូលាយ។

## អ្វីដែលអ្នកនឹងរៀន៖
1. **ការគ្រប់គ្រងសមម័យជាមួយគ្នា**៖ ប្រតិបត្តិការភ្នាក់ងារច្រើននៅក្នុងសម័យដាច់ដោយឡែក (លំដាប់ fan-out/fan-in)
2. **ConcurrentBuilder**៖ API ជាន់ខ្ពស់សម្រាប់សាងសង់ចរន្តការងារ concurrent
3. **សំណើផ្លូវដំណើរកំសាន្ត**៖ ភ្នាក់ងារពិសេសបីដែលធ្វើការជាមួយគ្នា
4. **ការប្រមូលផ្តុំលំនាំដើម**៖ ការរួមបញ្ចូលចម្លើយភ្នាក់ងារច្រើន
5. **អត្ថប្រយោជន៍ការសមត្ថភាព**៖ ការប្រតិបត្តិសមម័យជាមួយគ្នាចំពោះការដំណើរការតាមលំដាប់

## ភ្នាក់ងារពិសេសបី៖

1. **ភ្នាក់ងារទីតាំងទេសចរណ៍**៖ ទីតាំងទេសចរណ៍ សកម្មភាព អាគារសំខាន់ៗ
2. **ភ្នាក់ងារផឹកអាហារ**៖ ម្ហូបក្នុងតំបន់ ភោជនីយដ្ឋាន បទពិសោធន៍ផឹកអាហារ
3. **ភ្នាក់ងារជីវប្រវត្តិសាស្ត្រ**៖ ព័ត៌មានប្រវត្តិសាស្ត្រ អត្ថន័យវប្បធម៍ បរិបទ


In [ ]:
import asyncio
import json
import os
from typing import Any, cast

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("All imports successful!")

## ជំហានទី 1៖ កំណត់ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះកំណត់ស្កីម៉ាដែលនិទាឃរូបពិសេសនីមួយៗនឹងបញ្ចូនត្រឡប់មកវិញ។ នេះធ្វើឱ្យមានការឆ្លើយតបដែលមានតម្លាភាព និងអាចបកស្រាយបានពីភ្នាក់ងារទាំងអស់។


## ជំហានទី 1៖ កំណត់Model Pydantic សម្រាប់លទ្ធផលមានរចនាសម្ព័ន្ធ

Model ទាំងនេះកំណត់ស្នាដៃដែលភ្នាក់ងារប្រកបដោយជំនាញនីមួយៗនឹងត្រឡប់មកវិញ។ នេះធានាបានពីការឆ្លើយតបដែលមានរចនាសម្ព័ន្ធនិងអាចបកប្រែបានពីភ្នាក់ងារទាំងអស់។


In [ ]:
class AttractionsRecommendation(BaseModel):
    """Tourist attractions and activities recommendations."""

    destination: str
    top_attractions: list[str]
    activities: list[str]
    best_time_to_visit: str
    transportation_tips: str  


class DiningRecommendation(BaseModel):
    """Food and dining recommendations."""

    destination: str
    local_cuisine: str
    must_try_dishes: list[str]
    recommended_restaurants: list[str]
    food_experiences: list[str]
    dining_etiquette: str


class HistoryRecommendation(BaseModel):
    """Historical and cultural information."""

    destination: str
    historical_significance: str
    cultural_highlights: list[str]
    important_periods: list[str]
    cultural_experiences: list[str]
    interesting_facts: list[str]

## ជំហានទី 2៖ បញ្ចូលអថេរបរិស្ថាន និងកំណត់ Provider Foundry

ប្រើ `AzureAIProjectAgentProvider` ជាមួយការ Authentication keyless `AzureCliCredential` ដែលផ្ទៀងផ្ទាត់នឹងលំនាំដែលបានប្រើនៅមេរៀន 01–13។


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("Azure AI Foundry provider configured successfully!")

## ជំហាន 3៖ បង្កើតភ្នាក់ងារធ្វើដំណើរពិសេសបីរូប


In [ ]:
# Agent 1: Tourist Attractions Expert
attractions_agent = await provider.create_agent(
    name="attractions-agent",
    instructions=(
        "You are a tourism expert specializing in attractions and activities. "
        "When given a travel destination, provide comprehensive recommendations for "
        "tourist attractions, activities, best times to visit, and transportation tips. "
        "Focus on popular landmarks, unique experiences, and practical travel advice. "
        "Return structured JSON matching the AttractionsRecommendation schema."
    ),
)

# Agent 2: Food and Dining Expert
dining_agent = await provider.create_agent(
    name="dining-agent",
    instructions=(
        "You are a culinary expert specializing in local food and dining experiences. "
        "When given a travel destination, provide recommendations for local cuisine, "
        "must-try dishes, recommended restaurants, and unique food experiences. "
        "Include dining etiquette and cultural food customs. "
        "Return structured JSON matching the DiningRecommendation schema."
    ),
)


# Agent 3: History and Culture Expert
history_agent = await provider.create_agent(
    name="history-agent",
    instructions=(
        "You are a historian and cultural expert. "
        "When given a travel destination, provide historical context, cultural significance, "
        "important historical periods, cultural experiences, and interesting facts. "
        "Focus on helping travelers understand the cultural heritage and historical importance. "
        "Return structured JSON matching the HistoryRecommendation schema."
    ),
)

# ជំហានទី ៤៖ សាងសង់ដំណើរការជាក់ស្តែង

`WorkflowBuilder` ជាមួយកម្មវិធីអនុវត្តន៍តូចមួយនៃអ្នកចែកចាយ និង `add_fan_out_edges` ៖
1. **អ្នកចែកចាយ** ផ្សព្វផ្សាយបញ្ចូលដូចគ្នាទៅអោយភ្នាក់ងារមោងបីទាំងអស់
2. **ភ្នាក់ងារមោងបី** ដំណើរការដោយសម័យប្រព័ន្ធ
3. **លទ្ធផល** ប្រមូលចម្លើយនីមួយៗរបស់ភ្នាក់ងា្រក់ដោយប្លែកៗ


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [attractions_agent, dining_agent, history_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Concurrent Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Architecture:</strong><br>
        • Input → <strong>Dispatcher</strong> (fan-out)<br>
        • <strong>3 Agents</strong> run in parallel (attractions, dining, history)<br>
        • Output → 3 AgentResponse objects, one per agent
    </p>
</div>
"""))

## ជំហៀងទី 5: សំណើបំណែក 1 - ការផ្តល់អនុសាសន៍សម្រាប់ការធ្វើដំណើរទៅទីក្រុងតូក្យូ

ចូរយើងសាកល្បងដំណើរការប្រតិបត្តិការដូចគ្នារួមជាមួយទីក្រុងតូក្យូជាគោលដៅ។ អ្នកចូលរួមទាំងបីនាក់នឹងធ្វើការគ្នាដោយសម័យដើម្បីផ្តល់អនុសាសន៍ដំណើរការដំណើរកម្សាន្តយ៉ាងទូលំទូលាយ។


In [ ]:
async def display_travel_recommendations(destination: str):
    """Run the concurrent workflow and display formatted results."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Processing Travel Recommendations for {destination}</h3>
        <p style='margin: 0;'><strong>Status:</strong> Running 3 agents concurrently...</p>
    </div>
    """))

    # Run the workflow. With WorkflowBuilder(output_executors=[a1, a2, a3]),
    # outputs is a list of AgentResponse objects in the same order as output_executors.
    events = await workflow.run(f"I want comprehensive travel recommendations for {destination}")
    outputs = events.get_outputs()

    # Display results header
    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Complete Travel Guide for {destination}</h2>
        <p style='margin: 0; font-size: 14px; opacity: 0.9;'>Generated by 3 concurrent agents</p>
    </div>
    """))

    sections = [
        ("attractions-agent", AttractionsRecommendation, display_attractions_section),
        ("dining-agent", DiningRecommendation, display_dining_section),
        ("history-agent", HistoryRecommendation, display_history_section),
    ]

    for i, (agent_name, schema, render) in enumerate(sections):
        if i >= len(outputs):
            continue
        text = outputs[i].text
        try:
            data = schema.model_validate_json(text)
            render(data)
        except Exception as e:
            display(HTML(f"""
            <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>Error parsing {agent_name} response:</strong> {str(e)}
                <details><summary>Raw response</summary>{text}</details>
            </div>
            """))


def display_attractions_section(data: AttractionsRecommendation):
    """Display attractions recommendations in a formatted section."""
    attractions_list = "".join([f"<li>{attraction}</li>" for attraction in data.top_attractions])
    activities_list = "".join([f"<li>{activity}</li>" for activity in data.activities])

    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏛️ Tourist Attractions & Activities</h3>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Top Attractions:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{attractions_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
        <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Activities:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{activities_list}</ul>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Best Time to Visit:</strong> {data.best_time_to_visit}
        </div>
        <div>
            <strong style='color: #333;'>Transportation Tips:</strong> {data.transportation_tips}
        </div>
    </div>
    """))


def display_dining_section(data: DiningRecommendation):
    """Display dining recommendations in a formatted section."""
    dishes_list = "".join(
        [f"<li>{dish}</li>" for dish in data.must_try_dishes])
    restaurants_list = "".join(
        [f"<li>{restaurant}</li>" for restaurant in data.recommended_restaurants])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.food_experiences])

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🍜 Food & Dining Experiences</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Local Cuisine:</strong> {data.local_cuisine}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Must-Try Dishes:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{dishes_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Restaurants:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{restaurants_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Food Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <strong style='color: #333;'>Dining Etiquette:</strong> {data.dining_etiquette}
        </div>
    </div>
    """))


def display_history_section(data: HistoryRecommendation):
    """Display history recommendations in a formatted section."""
    highlights_list = "".join(
        [f"<li>{highlight}</li>" for highlight in data.cultural_highlights])
    periods_list = "".join(
        [f"<li>{period}</li>" for period in data.important_periods])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.cultural_experiences])
    facts_list = "".join(
        [f"<li>{fact}</li>" for fact in data.interesting_facts])

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>📚 History & Culture</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Historical Significance:</strong> {data.historical_significance}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Highlights:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{highlights_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Important Historical Periods:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{periods_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Interesting Facts:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{facts_list}</ul>
        </div>
    </div>
    """))


# Test with Tokyo
await display_travel_recommendations("Tokyo")

# ជំហានទី 6: សំណួរពិសោធន៍ទី 2 - ការផ្តល់អនុសាសន៍ធ្វើដំណើរទៅទីក្រុងប៉ារីស


In [ ]:
await display_travel_recommendations("Paris")

## ជំហានទី 7: វិភាគសមត្ថភាព - សម័យសមConcurrent ទៅ Sequential

យើងមកវាស់វែងភាពខុសគ្នានៃសមត្ថភាពរវាងការប្រតិបត្តិដំណើរការជាសម័យសម Concurrent និងជាតម្រៀប ដើម្បីបង្ហាញអត្ថប្រយោជន៍នៃការគ្រប់គ្រងសម័យសម Concurrent។


In [ ]:
import time


async def measure_concurrent_performance(destination: str):
    """Measure concurrent execution time."""
    start_time = time.time()

    events = await workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs)


async def measure_sequential_performance(destination: str):
    """Measure sequential execution time."""
    # Build a sequential workflow that chains the same agents one after another.
    sequential_workflow = (
        WorkflowBuilder(
            start_executor=attractions_agent,
            output_executors=[attractions_agent, dining_agent, history_agent],
        )
        .add_chain([attractions_agent, dining_agent, history_agent])
        .build()
    )
    start_time = time.time()

    events = await sequential_workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs)


async def performance_comparison():
    """Compare concurrent vs sequential performance."""
    test_destination = "Barcelona"

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Performance Comparison Test</h3>
        <p style='margin: 0;'>Testing with destination: <strong>Barcelona</strong></p>
    </div>
    """))

    # Test concurrent execution
    print("Running concurrent workflow...")
    concurrent_time, concurrent_count = await measure_concurrent_performance(test_destination)

    # Test sequential execution
    print("Running sequential workflow...")
    sequential_time, sequential_count = await measure_sequential_performance(test_destination)

    # Calculate performance improvement
    improvement = ((sequential_time - concurrent_time) / sequential_time) * 100

    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(102,126,234,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Performance Results</h2>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>⚡ Concurrent Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{concurrent_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{concurrent_count} agent responses</p>
            </div>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>🔄 Sequential Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{sequential_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{sequential_count} agent responses</p>
            </div>
        </div>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Performance Improvement</h4>
            <p style='margin: 0; font-size: 20px; font-weight: bold;'>{improvement:.1f}% faster</p>
            <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>
                Saved {sequential_time - concurrent_time:.2f} seconds with concurrent execution
            </p>
        </div>
    </div>
    """))


# Run performance comparison
await performance_comparison()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
